## Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import kaggle
import csv
import os
import nltk
from nltk.corpus import wordnet as wn
import random
from transformers import BertTokenizer, BertModel
import torch 


/Users/id06/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/id06/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cleaning Data

In this section, we first download the data, then reformat the initial files by getting rid off unneeded columns in certain rows which may lead to errors. In addition, the words are all put into one file "words.csv" making them easier to deal with in the next part.

In [ ]:
kaggle.api.authenticate()

dataset = "thedevastator/common-english-parts-of-speech"
files = ["adjectives.csv", "adverbs.csv", "nouns.csv", "verbs.csv"]

for file in files:
    kaggle.api.dataset_download_file(
        dataset,
        file_name=file,
        path="Data/"
    )
    
    # files download as .zip so unzip them
    zip_path = f"Data/{file}.zip"
    if os.path.exists(zip_path):
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall("Data/")
        os.remove(zip_path)

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /Users/id06/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
files = ['Data/adjectives.csv', 'Data/adverbs.csv', 'Data/nouns.csv', 'Data/verbs.csv']

# This goes through a file and keeps just the first column of each one
def clean_csv(input_file):
    temp_file = input_file + ".csv"
    with open(input_file, "r", newline="", encoding="utf-8") as infile, \
        open(temp_file, "w", newline="", encoding="utf-8") as outfile:

        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        for row in reader:
            if row:
                writer.writerow([row[0]])

    os.replace(temp_file, input_file)

# This takes the files and turns them into a single csv file with two columns, one for the word and one for the part of speech
def files_to_csv(files):
    df = pd.DataFrame()

    for file in files:
        clean_csv(file)
        pos = os.path.basename(file).split('.')[0]
        temp_df = pd.read_csv(file, header=None, names=['word'])
        temp_df['pos'] = pos
        df = pd.concat([df, temp_df], ignore_index=True)

    df.to_csv('Data/words.csv', index=False)

# Executing the functions on "files" list
files_to_csv(files)

### Removing Unneeded and Weird Words

This is being done to preseve a balance between the numbers of words in each class as well as removing strange words (some of which, especially for nouns, are just strings of numbers).

In [7]:
# This function reads the csv and deletes rows where the word contains either a space a or a digit, since we only want single words
def clean_words(input_file):
    df = pd.read_csv(input_file)
    df = df.dropna(subset=['word'])
    df = df[~df['word'].str.contains(' ')]
    df = df[~df['word'].str.contains(r'-')]
    df = df[~df['word'].str.contains(r'\d', regex=True)]
    df.to_csv(input_file, index=False)

clean_words('Data/words.csv')



### Stabalizing Classes

Because there is a lot of imbalance in the number of words in each class, I needed to get rid of some and add other. To do this I added verbs from the NLTK library "wordnet" because my original dataset had only 125 or so. I also, for the other parts of speech, randomly sampled from these classes so as to only take as many as I needed.

In [8]:
# adding verbs to words.csv
verbs = set()
for synset in wn.all_synsets(pos=wn.VERB):
    for lemma in synset.lemmas():
        verbs.add(lemma.name())

verbs = [v for v in verbs if '_' not in v]

print(f"Number of verbs from WordNet: {len(verbs)}")

words = pd.read_csv('Data/words.csv')
print("csv length before adding verbs:", len(words))

for word in verbs:
    if word not in words['word'].values:
        words.loc[len(words)] = {'word': word, 'pos': 'verbs'}
words.to_csv('Data/words.csv', index=False)

# This reads the csv and evens out the number of words in each pos by randomly sampling from each
words = pd.read_csv('Data/words.csv')

min_count = words['pos'].value_counts().min()

# Sampling from each one
final_df = (
    words
    .groupby('pos', group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=42))
)

final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv('Data/words.csv', index=False)


Number of verbs from WordNet: 8702
csv length before adding verbs: 137822


/var/folders/w0/w0zxtn3j7675228kq6fxwb2r0000gn/T/ipykernel_9195/4148829197.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  words


### Removing Words Existing as Multiple Classes

If one word such as "walk" exists as both a verb (e.x. 'I will walk') and noun (e.x. 'I will go for a walk'), I will remove this to reduce model confusion.

In [9]:
import pandas as pd

df = pd.read_csv('Data/words.csv')

print("csv length before removing duplicates:", len(df))

df = df[~df['word'].duplicated(keep=False)]
df.to_csv('Data/words.csv', index=False)

print("csv length after removing duplicates:", len(df))

csv length before removing duplicates: 18392
csv length after removing duplicates: 18105


### Words to Embeddings

In this section we take each of the words and map them to embeddings which we will use later on for our machine learning process and catagorizing words based on part of speech. I am using the BERT model by Google from HuggingFace to do tokenization and create embeddings for each of the words. Also, padding of vectors of zeroes were added to each word to make sure that the length of each of the words were the same and could be handled by the model

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

df_words = pd.read_csv('Data/words.csv')
data = []

for index, row in df_words.iterrows():
    word = row['word']
    pos = row['pos']
    
    inputs = tokenizer(word, return_tensors='pt')

    with torch.no_grad():
        embeddings = model.embeddings(
            input_ids=inputs['input_ids'],
            token_type_ids=inputs['token_type_ids']
        )

    embeddings = embeddings[0][:-1]  # keep CLS but drop SEP tokens
    data.append({'word': word, 'pos': pos, 'embedding': embeddings})


max_len = max(item['embedding'].shape[0] for item in data)

for item in data:
    seq_len = item['embedding'].shape[0]
    padding_length = max_len - seq_len
    
    mask = torch.zeros(max_len, dtype=torch.bool)
    mask[seq_len:] = True
    item['padding_mask'] = mask
    
    if padding_length > 0:
        padding = torch.zeros((padding_length, item['embedding'].shape[1]))
        item['embedding'] = torch.cat((item['embedding'], padding), dim=0)

torch.save(data, 'Data/word_embeddings.pt')